In [ ]:

#* Se requiere la intalación de las siguientes librerías:
#* - langchain langchain-text-splitters langchain-community bs4
#* - langchain-[model]
#* - langchain-chroma
#* - pypdf
#* Revisar la documentación https://docs.langchain.com/oss/python/langchain/rag

In [ ]:

#* importaciones necesarias
from pathlib import Path

#* Cargador de documentos PDF
from langchain_community.document_loaders import PyPDFLoader

#* División de texto en fragmentos 
from langchain_text_splitters import RecursiveCharacterTextSplitter

#* Modelos de Ollama
from langchain_ollama import ChatOllama, OllamaEmbeddings

#* Vector store basado en Chroma
from langchain_chroma import Chroma

#* Prompting y construcción de la cadena RAG 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

In [2]:
BASE_DIR = Path.cwd().parent.parent
DATA_DIR = BASE_DIR / "assets"
ENV_DIR  = BASE_DIR / ".env"

#* PDF para el ejemplo
PDF_PATH = DATA_DIR / "terminosycondicionestodoclaro.pdf"

In [3]:
#* Creamos el loader para PDF
loader = PyPDFLoader(str(PDF_PATH))

documents = loader.load()

print(f"Número de páginas cargadas desde el PDF: {len(documents)}")
print("Ejemplo de contenido (primeros 500 caracteres de la primera página):\n")
print(documents[0].page_content[:500], "...")

Número de páginas cargadas desde el PDF: 8
Ejemplo de contenido (primeros 500 caracteres de la primera página):

1 
   
 
Documento Claro Colombia 
Términos y Condiciones del beneficio Todo Claro 
 
 
✓ Oferta válida del 01 de agosto al 31 de agosto de 2024. 
 
• CONDICIONES GENERALES:  
 
✓ Oferta para usuarios de Comcel S.A. dirigida únicamente a personas naturales que sean clientes 
nuevos y/o actuales con tarifas residenciales o masivas. 
✓ No aplica para empresas y/o negocios. 
✓ Aplica para planes postpago móvil con voz y datos y planes hogar que tengan servicio de internet fijo.  
✓ Con esta promoci ...


In [4]:

#* Dividimos el texto en fragmentos manejables
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, #* Tamaño aproximado de cada fragmento en caracteres
    chunk_overlap=200, #* Superposición entre fragmentos para mantener contexto
    add_start_index=True 
)

#* Aplicamos el splitter a la lista de documentos cargados
splits = text_splitter.split_documents(documents)

print(f"Número de fragmentos creados: {len(splits)}")
print("Ejemplo de fragmento:\n")
print(splits[0].page_content[:500], "...")


Número de fragmentos creados: 22
Ejemplo de fragmento:

1 
   
 
Documento Claro Colombia 
Términos y Condiciones del beneficio Todo Claro 
 
 
✓ Oferta válida del 01 de agosto al 31 de agosto de 2024. 
 
• CONDICIONES GENERALES:  
 
✓ Oferta para usuarios de Comcel S.A. dirigida únicamente a personas naturales que sean clientes 
nuevos y/o actuales con tarifas residenciales o masivas. 
✓ No aplica para empresas y/o negocios. 
✓ Aplica para planes postpago móvil con voz y datos y planes hogar que tengan servicio de internet fijo.  
✓ Con esta promoci ...


In [5]:

#* Modelo de embeddings basado en HuggingFace
model_embedding = OllamaEmbeddings(model="embeddinggemma:300m")

print("Objeto de embeddings creado correctamente.")

Objeto de embeddings creado correctamente.


In [6]:

#* Directorio donde se va almacenar la base vectorial
CHROMA_DIR = BASE_DIR / "chroma_db"
persist_dir = str(CHROMA_DIR / "db_todo_claro_langchain")

#* Creamos el vector store vacío
vector_store = Chroma(
    collection_name="todo_claro_collection",
    embedding_function=model_embedding,
    persist_directory=persist_dir
)

#* Indexamos (añadimos) todos los fragmentos al vector store
documents_ids = vector_store.add_documents(splits)

print(f"Se indexaron {len(documents_ids)} fragmentos en Chroma.")
print(f"La base vectorial se guardó en la carpeta: {persist_dir}")

Se indexaron 22 fragmentos en Chroma.
La base vectorial se guardó en la carpeta: d:\nalvarez\100_cursos\bases_vectoriales\chroma_db\db_todo_claro_langchain


In [7]:

#* Creación de retriever a partir del vector store
retriever_todo_claro = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

print("Retriever creado correctamente a partir del vector store.")

Retriever creado correctamente a partir del vector store.


In [17]:

#* Cargamos el modelo de Ollama a usar
model_llm = ChatOllama(
    model="gemma3:4b",
    temperature=0.3
)

#* Prompt siguiendo la filosofía RAG
template = """
Eres un asistente especializado en términos y condiciones de productos de telecomunicaciones.

Uso EXCLUSIVAMENTE la información del contexto para responder en español latinoamericano y en un máximo de 4 párrafos. Si la pregunta no se puede responder con el contexto, di claramente que la información no esta en el documento.

Pregunta del usuario: 
{question}

Contexto:
{context}
"""

prompt = ChatPromptTemplate.from_template(template)

print("Modelo de chat y prompt RAG configurados.")

Modelo de chat y prompt RAG configurados.


In [18]:

#* Creamos la cadena RAG

rag_chain = (
    {
        "question": RunnablePassthrough(),
        "context": retriever_todo_claro,
    }
    | prompt
    | model_llm
)

print("Cadena RAG creada correctamente.")

Cadena RAG creada correctamente.


In [19]:
rag_chain

{
  question: RunnablePassthrough(),
  context: VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x0000023AA21B1430>, search_kwargs={'k': 4})
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nEres un asistente especializado en términos y condiciones de productos de telecomunicaciones.\n\nUso EXCLUSIVAMENTE la información del contexto para responder en español latinoamericano y en un máximo de 4 párrafos. Si la pregunta no se puede responder con el contexto, di claramente que la información no esta en el documento.\n\nPregunta del usuario: \n{question}\n\nContexto:\n{context}\n'), additional_kwargs={})])
| ChatOllama(model='gemma3:4b', temperature=0.3)

In [20]:

#* Probamos la cadena RAG con una pregunta de ejemplo

pregunta = (
    "Según los términos y condiciones del beneficio Todo Claro, "
    "¿en qué casos un cliente NO puede acceder temporalmente al beneficio "
    "para sus servicio hogar Claro?"
)

respuesta = rag_chain.invoke(pregunta)

print("Pregunta:")
print(pregunta)
print("\nRespuesta generada por el modelo:")
print(respuesta.content)

Pregunta:
Según los términos y condiciones del beneficio Todo Claro, ¿en qué casos un cliente NO puede acceder temporalmente al beneficio para sus servicio hogar Claro?

Respuesta generada por el modelo:
Okay, here's a breakdown of the key information extracted from the provided documents, organized for clarity:

**Core Benefit: "Todo Claro" Benefit**

*   **Availability:** The "Todo Claro" benefit (likely referring to increased data or internet speeds) is available under specific conditions.
*   **Eligibility:**
    *   **Plans:** Applies to internet plans starting at 30 megabytes (Mbps) in WTTH (Wide-Area Transmission Technology Home) coverage.  Does *not* apply to speeds of 500 Mbps or higher in HFC and FTTH (Fiber-to-the-Home) coverage.
    *   **Programs:**  Applies to users participating in the Ministry of Information and Communication (TIC) incentive program.
*   **Activation Requirements:**
    *   **Matching Data:** The installation/activation address for the mobile post-paid 

In [13]:

#* Visualizamos las fuentes utilizadas para responder
print("Pregunta de prueba:")
print(pregunta)
print("\nFragmentos recuperados:\n")
docs_relevantes = retriever_todo_claro.invoke(pregunta)
print(f"Se recuperaron {len(docs_relevantes)} fragmentos relevantes:\n")
for i, doc in enumerate(docs_relevantes, 1):
    print(f"Fragmento {i}:\n{doc.page_content}\n{'-'*50}\n")

Pregunta de prueba:
Según los términos y condiciones del beneficio Todo Claro, ¿en qué casos un cliente NO puede acceder temporalmente al beneficio para sus servicio hogar Claro?

Fragmentos recuperados:

Se recuperaron 4 fragmentos relevantes:

Fragmento 1:
✓ El beneficio de Todo Claro no aplica para servicios de cortesía, demo, Inhouse, o planes especiales con 
renta cero, postpago y/o Hogar. 
✓ El beneficio de Todo Claro no aplica para planes de datos para servicios especiales como SMARTWATCH, 
vehículos y/o otros servicios de datos que no estén diseñados para smartphones. 
✓ Los clientes que tengan el beneficio Todo Claro en su línea móvil postpago, podrán disfrutar de la 
navegación a la aplicación Claro vídeo sin límite de consumo de datos del plan. 
✓ Para conocer los beneficios, el cliente puede hacerlo con el asesor comercial en la compra de los nuevos 
productos y/o poniéndose en contacto con las líneas de servicio.
--------------------------------------------------

Fragment